# 🇵🇭 Sugidanon — Text → Hiligaynon Translator

Team Hague · June 25–26, 2026

Type English/Tagalog → get Hiligaynon. Two modes:
- **Neural** (fluent): a Hiligaynon LLM. Needs a GPU runtime + model download.
- **Dictionary** (instant fallback): offline word-by-word, no GPU.

**For GPU:** Runtime → Change runtime type → Hardware accelerator → **GPU**, then Run all.

## 1. Clone repo

In [ ]:
import os
if not os.path.isdir('tinig-sa-liwanag'):
    !git clone https://github.com/Jazztinn/tinig-sa-liwanag.git
%cd tinig-sa-liwanag

## 2. Dictionary mode (works immediately, no install)

In [ ]:
!python scripts/translate_hil.py "Good morning, I went to the market yesterday"

## 3. Neural mode — install + load model once
First run downloads the model (a few minutes). GPU strongly recommended.
Swap `MODEL` for any Hiligaynon LLM from `RESOURCES.md`.

In [ ]:
!pip -q install transformers accelerate torch sentencepiece

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Tried in order; first that loads wins. Add/reorder freely (RESOURCES.md).
CANDIDATES = [
    "welyjesch/lfm25-sft-hiligaynon",
    "PLTAT/hiligaynon_llama_3.1_finetuned_lora",
]

tok = model = MODEL = None
for name in CANDIDATES:
    try:
        print(f"Loading {name} ... (first time downloads weights)")
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(
            name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        MODEL = name
        print("Loaded:", name, "| device:", model.device)
        break
    except Exception as e:
        print(f"  could not load {name}: {type(e).__name__}: {e}\n  trying next...")

if model is None:
    print("\nNo neural model loaded (gated/offline?). "
          "Use the dictionary backend in section 2 instead:")
    print('  !python scripts/translate_hil.py "your text"')

In [ ]:
def translate(text, max_new_tokens=128):
    if model is None:
        return "(no neural model — use dictionary backend)"
    instruction = (
        "Translate the following text into Hiligaynon (Ilonggo). "
        "Reply with only the Hiligaynon translation.\n\n"
        f"Text: {text}\nHiligaynon:"
    )
    # Use the model's chat template if it has one (better for instruct models),
    # else fall back to a plain prompt.
    if getattr(tok, "chat_template", None):
        inputs = tok.apply_chat_template(
            [{"role": "user", "content": instruction}],
            add_generation_prompt=True, return_tensors="pt",
        ).to(model.device)
        gen_kwargs = {"max_new_tokens": max_new_tokens, "do_sample": False}
        with torch.no_grad():
            out = model.generate(inputs, **gen_kwargs)
        decoded = tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
        return decoded.strip()
    inputs = tok(instruction, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tok.decode(out[0], skip_special_tokens=True)
    return decoded.split("Hiligaynon:")[-1].strip()

# quick test
for s in ["Good morning, how are you?", "I went to the market yesterday",
          "Thank you very much for your help"]:
    print(f"{s}\n  -> {translate(s)}\n")

## 4. Interactive — type your own

In [ ]:
while True:
    text = input("EN/TL (blank to stop): ").strip()
    if not text:
        break
    print("  HIL:", translate(text), "\n")

## 5. (Optional) Interactive UI with a text box

In [ ]:
import ipywidgets as widgets
from IPython.display import display

box = widgets.Text(description="EN/TL:", layout=widgets.Layout(width="70%"))
out = widgets.Output()

def on_submit(change):
    with out:
        out.clear_output()
        print("HIL:", translate(box.value))

box.on_submit(on_submit)
display(box, out)